# MoodNote-AI — Fine-tune ViSoBERT trên Google Colab

Pipeline phân loại cảm xúc tiếng Việt (7 classes) với **ViSoBERT** + UIT-VSMEC + ViGoEmotions.

| Bước | Cell | Mô tả |
|------|------|-------|
| Setup | 1 | GPU, Drive, Clone repo, Cài dependencies |
| Download | 2 | UIT-VSMEC + ViGoEmotions |
| Data Prep | 3 | Merge, Preprocess (word segmentation) |
| Augmentation | 4 | Random aug + Back-translation (Surprise, Anger) |
| Train | 5 | Curriculum Learning: Stage 1 → Stage 2 |
| Evaluate | 6 | Test predictions & classification report |

> **Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── GPU Check ─────────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. Runtime -> Change runtime type -> T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Mount Google Drive ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Clone Repo & Install ───────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = 'https://github.com/ToanHuynh0201/MoodNote-AI.git'  # <- thay bang URL repo cua ban
REPO_DIR = '/content/MoodNote-AI'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)

os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
subprocess.run(['pip', 'install', 'deep_translator', '-q'], check=True)
sys.path.insert(0, REPO_DIR)

# ── Paths ──────────────────────────────────────────────────────────────────────
CONFIG_DIR     = f'{REPO_DIR}/configs'
RAW_DIR        = f'{REPO_DIR}/data/raw'
PROCESSED_DIR  = f'{REPO_DIR}/data/processed'
DRIVE_DIR      = '/content/drive/MyDrive/MoodNote-AI'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
BEST_MODEL_DIR = f'{DRIVE_DIR}/best_model'

for d in [RAW_DIR, PROCESSED_DIR, CHECKPOINT_DIR, BEST_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("\nSetup hoan tat.")
print(f"  Repo    : {REPO_DIR}")
print(f"  Drive   : {DRIVE_DIR}")

In [ ]:
import os
os.chdir(REPO_DIR)

# ── UIT-VSMEC ──────────────────────────────────────────────────────────────────
print("=" * 50)
print("Downloading UIT-VSMEC...")
from src.data.download_dataset import download_uit_vsmec
download_uit_vsmec(output_dir=RAW_DIR)

# ── ViGoEmotions (gated - can HF Token) ───────────────────────────────────────
# Truoc khi chay: Runtime -> Manage secrets -> them key HF_TOKEN
print("\n" + "=" * 50)
print("Downloading ViGoEmotions...")
from google.colab import userdata
from src.data.download_vigoemotions import download_vigoemotions
hf_token = userdata.get('HF_TOKEN')
# output_dir=RAW_DIR vi ham tu append /vigoemotions ben trong
download_vigoemotions(output_dir=RAW_DIR, token=hf_token)

print("\nDownload hoan tat.")

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.merge_datasets import main as merge_main
from src.data.preprocess import preprocess_dataset

# ── Merge VSMEC + ViGoEmotions ─────────────────────────────────────────────────
print("=" * 50)
print("Merging VSMEC + ViGoEmotions...")
merge_main(
    vsmec_dir=RAW_DIR,
    vigoemotions_dir=f'{RAW_DIR}/vigoemotions',
    output_dir=f'{REPO_DIR}/data/merged'
)

# ── Preprocess merged (train + val + test) ─────────────────────────────────────
print("\n" + "=" * 50)
print("Preprocessing merged dataset...")
preprocess_dataset(
    input_dir=f'{REPO_DIR}/data/merged',
    output_dir=PROCESSED_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

# ── Preprocess VSMEC-only (dung cho Stage 1) ──────────────────────────────────
print("\n" + "=" * 50)
print("Preprocessing VSMEC-only (Stage 1)...")
VSMEC_ONLY_DIR = f'{PROCESSED_DIR}/vsmec_only'
preprocess_dataset(
    input_dir=RAW_DIR,
    output_dir=VSMEC_ONLY_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

print("\nData prep hoan tat.")
# ── Deduplicate va resplit (80/10/10 stratified) ─────────────────────────────
print("
" + "=" * 50)
print("Deduplicating and resplitting...")
import subprocess
subprocess.run(["python", f"{REPO_DIR}/scripts/resplit_stratified.py"], check=True)


In [ ]:
import os, time
import pandas as pd
os.chdir(REPO_DIR)

from src.data.augment import augment_dataset, VietnameseAugmenter

AUGMENTED_TRAIN = f'{PROCESSED_DIR}/train_augmented.csv'

# ── Random Augmentation (Anger, Fear, Disgust, Surprise) ──────────────────────
print("=" * 50)
print("Random augmentation (deletion / swap / insertion)...")
augment_dataset(
    input_csv=f'{PROCESSED_DIR}/train.csv',
    output_csv=AUGMENTED_TRAIN,
    target_counts={2: 1800, 3: 1400, 4: 1300, 5: 1300},  # Surprise: 2000→1300 (tránh noise ratio quá cao)
    techniques=["deletion", "swap", "insertion"],
    seed=42
)

# ── Back-Translation helper ────────────────────────────────────────────────────
def run_backtranslation(df, label_id, n_samples, label_name):
    aug = VietnameseAugmenter(seed=42)
    samples = df[df['label'] == label_id].head(n_samples)
    bt_rows = []
    for i, (_, row) in enumerate(samples.iterrows()):
        bt = aug.back_translate(row['text'])
        if bt != row['text'] and bt.strip():
            bt_rows.append({'text': bt, 'label': label_id})
        time.sleep(0.1)
        if (i + 1) % 50 == 0:
            print(f"  {label_name}: {i + 1}/{n_samples} — {len(bt_rows)} valid")
    return bt_rows

augmented_df = pd.read_csv(AUGMENTED_TRAIN)

# ── Back-Translation: Surprise (class 5) ──────────────────────────────────────
print("\n" + "=" * 50)
print("Back-translation: Surprise (class 5, 400 samples)...")  # tăng từ 200→400 (back-translation bảo toàn ngữ nghĩa)
bt_surprise = run_backtranslation(augmented_df, label_id=5, n_samples=400, label_name="Surprise")
print(f"  Surprise: {len(bt_surprise)} paraphrases")

# ── Back-Translation: Anger (class 2) ─────────────────────────────────────────
print("\n" + "=" * 50)
print("Back-translation: Anger (class 2, 150 samples)...")
bt_anger = run_backtranslation(augmented_df, label_id=2, n_samples=150, label_name="Anger")
print(f"  Anger: {len(bt_anger)} paraphrases")

# ── Save ───────────────────────────────────────────────────────────────────────
all_bt = bt_surprise + bt_anger
if all_bt:
    bt_df = pd.DataFrame(all_bt)
    final_df = pd.concat([augmented_df, bt_df], ignore_index=True).sample(frac=1, random_state=42)
    final_df.to_csv(AUGMENTED_TRAIN, index=False)
    print(f"\nDataset sau augmentation: {len(final_df)} mau")
    print(final_df['label'].value_counts().sort_index().to_string())
else:
    print("\nKhong co BT paraphrase — giu nguyen random augmented dataset.")

In [ ]:
import os, torch
import pandas as pd
os.chdir(REPO_DIR)

from src.data.dataset import EmotionDataset
from src.models.phobert_classifier import PhoBERTEmotionClassifier
from src.models.model_utils import save_model, get_device, print_model_summary
from src.training.trainer import train_model
from src.utils.config import load_all_configs
from src.utils.logger import setup_logger
from src.utils.metrics import compute_metrics, print_metrics, plot_confusion_matrix
import numpy as np
from pathlib import Path

logger = setup_logger(name='moodnote')

# Load configs
configs         = load_all_configs(CONFIG_DIR)
model_config    = configs['model']
training_config = configs['training']

logger.info(f"Model         : {model_config['model']['name']}")
logger.info(f"Epochs        : {training_config['training']['num_epochs']}")
logger.info(f"Batch size    : {training_config['training']['batch_size']}")
logger.info(f"LR            : {training_config['training']['learning_rate']}")
logger.info(f"Focal gamma   : {model_config['model'].get('focal_gamma', 2.0)}")

device = get_device()

model_name = model_config['model']['name']
max_len    = model_config['model']['max_seq_length']

# Smoke test: xac nhan dung model ViSoBERT
from transformers import AutoModel as _AM
_m = _AM.from_pretrained(model_name)
assert _m.config.hidden_size == 768, f"Expected hidden_size=768, got {_m.config.hidden_size}"
assert _m.config.num_hidden_layers == 12, f"Expected 12 layers, got {_m.config.num_hidden_layers}"
del _m
logger.info("Smoke test passed: hidden_size=768, num_hidden_layers=12")

# Datasets — dung merged train (VSMEC + ViGoEmotions, khong augment random)
MERGED_TRAIN = f'{PROCESSED_DIR}/train.csv'

train_dataset = EmotionDataset(MERGED_TRAIN,                      tokenizer_name=model_name, max_length=max_len)
val_dataset   = EmotionDataset(f'{PROCESSED_DIR}/validation.csv', tokenizer_name=model_name, max_length=max_len)
test_dataset  = EmotionDataset(f'{PROCESSED_DIR}/test.csv',       tokenizer_name=model_name, max_length=max_len)

logger.info(f"Train (merged): {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Class weights tinh tren merged train
train_labels  = pd.read_csv(MERGED_TRAIN)['label'].tolist()
class_counts  = np.bincount(train_labels, minlength=7).astype(float)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 7
print("Class weights (merged train):")
for i, w in enumerate(class_weights):
    print(f"  {model_config['emotion_labels'][i]:<12}: {w:.3f}")

# Model
model = PhoBERTEmotionClassifier(
    model_name=model_name,
    num_labels=model_config['model']['num_labels'],
    dropout=model_config['model']['dropout'],
    class_weights=class_weights,
    label_smoothing=model_config['model'].get('label_smoothing', 0.0),
    focal_gamma=model_config['model'].get('focal_gamma', 2.0)
)
model.to(device)
print_model_summary(model)

if torch.cuda.is_available():
    logger.info(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# ── Single-stage training: merged data, no R-Drop ─────────────────────────────
print("\n" + "=" * 50)
print("Training: merged VSMEC + ViGoEmotions — 20 epochs (no R-Drop, no augmentation)")
print("=" * 50)

training_config['training']['rdrop_alpha']   = 0.0   # khong R-Drop
training_config['training']['num_epochs']    = 20
training_config['training']['warmup_ratio']  = 0.1

trainer = train_model(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_config=training_config,
    output_dir=CHECKPOINT_DIR,
    use_wandb=training_config.get('wandb', {}).get('enabled', False)
)

# Evaluate on test set
logger.info("Evaluating on test set...")
predictions = trainer.predict(test_dataset)
detailed    = compute_metrics(predictions.predictions, predictions.label_ids)
print_metrics(detailed, model_config['emotion_labels'])

# Confusion matrix
plot_confusion_matrix(
    predictions.predictions, predictions.label_ids,
    emotion_labels=model_config['emotion_labels'],
    save_path=Path(CHECKPOINT_DIR) / 'confusion_matrix.png'
)

# Save
save_model(
    model=trainer.model,
    tokenizer=train_dataset.tokenizer,
    save_dir=BEST_MODEL_DIR,
    config={
        'model_config': model_config,
        'training_config': training_config,
        'test_results': {
            'accuracy':    detailed['accuracy'],
            'f1_macro':    detailed['f1_macro'],
            'f1_weighted': detailed['f1_weighted']
        }
    }
)

print("\n" + "=" * 50)
print("TRAINING HOAN TAT")
print("=" * 50)
print(f"Accuracy   : {detailed['accuracy']:.4f}")
print(f"F1-Macro   : {detailed['f1_macro']:.4f}")
print(f"F1-Weighted: {detailed['f1_weighted']:.4f}")


In [ ]:
import os
os.chdir(REPO_DIR)

from src.inference.predictor import EmotionPredictor

# Files trong Drive
print("Files trong best_model:")
for f in sorted(os.listdir(BEST_MODEL_DIR)):
    size = os.path.getsize(f'{BEST_MODEL_DIR}/{f}') / 1024**2
    print(f"  {f:<30} {size:.1f} MB")

# Test predict
print("\nTest predict:")
predictor = EmotionPredictor(model_path=BEST_MODEL_DIR)

test_sentences = [
    "Hôm nay tôi rất vui vì được nghỉ học!",          # Enjoyment
    "Tôi buồn quá, không biết phải làm sao.",          # Sadness
    "Thật tức giận khi bị đối xử bất công.",           # Anger
    "Trời ơi, tin này làm tôi bất ngờ quá!",           # Surprise
    "Tôi sợ lắm, không dám đi một mình.",              # Fear
    "Thấy ghê quá, không thể chịu được.",              # Disgust
]

for sentence in test_sentences:
    result = predictor.predict(sentence)
    print(f"  Text      : {sentence}")
    print(f"  Emotion   : {result['emotion']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print()